In [6]:
import numpy as np
import pandas as pd
import random
# load et simulation function from simulate_population.py: 
from simulate_population import sim_population

## 1 Simulate data

In [4]:
pop = sim_population(N=20000, step_forward=2, randomseed=42)
pop.step() # move +2y
pop.step() # move + 2y
pop.step() #move + 2y
pop.step() #move + 2y
pop.step() #move + 2y

In [532]:
pop.history[1].head()

,id,start,end,age_start,bmi,hyp,smoke,sex,eth,first_a,...,time_a,time_b,time_c,time_d,time_e,event_a,event_b,event_c,event_d,event_e
0,1,2,4,62.1,-1.8,0,0,1,0,NaN,...,2.080,0.264,1.118,55.531,52.353,0,1,1,0,0
1,2,2,4,43.0,-0.4,0,0,1,2,2.172,...,0.172,2.035,17.578,47.791,19.104,1,0,0,0,0
2,3,2,4,66.9,-1.7,0,0,0,0,2.569,...,0.569,0.153,0.012,48.803,65.261,1,1,1,0,0
3,4,2,4,57.7,2.0,1,0,0,2,0.100,...,0.178,0.342,0.303,51.891,272.711,1,1,1,0,0
4,5,2,4,23.4,0.2,0,0,1,0,NaN,...,27.104,4.313,43.508,2.099,7.984,0,0,0,0,0


In [511]:
#a single long dataframe - if needed
#long_df = pd.concat(pop.history, ignore_index=True)
#long_df = long_df.sort_values(by=['id', "start"])
#long_df.head(10)

### b) Create df_long 
	id	age_start	bmi	hyp	smoke	sex	eth	event	time	age
	2	43.0	-0.4	0	0	1	2	first_a	2.172	45.172
	3	66.9	-1.7	0	0	0	0	first_a	2.569	69.469

In [560]:
event_cols = ["first_a", "first_b", "first_c", "first_d", "first_e"]
context_cols = ['age_start', 'bmi', 'hyp', 'smoke', 'sex', 'eth']
df_short = pop.history[5][["id"]+context_cols + event_cols].copy()
df_short["first_alive"] = 0
df_long = df_short.melt(id_vars = ["id"] + context_cols, 
                       value_vars = event_cols + ["first_alive"], 
                       var_name = "event", 
                       value_name = "time").dropna(subset=["time"])
df_long["age"] = df_long["age_start"] + df_long["time"]
df_long = df_long.sort_values(by=['id', "age"])
df_long.head()

,id,age_start,bmi,hyp,smoke,sex,eth,event,time,age
100000,1,62.1,-1.8,0,0,1,0,first_alive,0.000,62.100
20000,1,62.1,-1.8,0,0,1,0,first_b,0.059,62.159
40000,1,62.1,-1.8,0,0,1,0,first_c,3.118,65.218
100001,2,43.0,-0.4,0,0,1,2,first_alive,0.000,43.000
1,2,43.0,-0.4,0,0,1,2,first_a,2.172,45.172


## 2 Check CoxPH

* eth_beta = np.array([0 if (x==0) else 0.5 if (x==1) else 2 for x in df["eth"]])
* chances are higher if there was a heart failure before 
* eventb_beta = (~df["first_b"].isna()).astype(int) * 0.3
* lp1 = 0.1*np.exp(0.5*df.bmi + 0.7*df.hyp + 0.4*(df.age-50)/15 + eth_beta + eventb_beta)
* time_a = 0.01 + np.round(rng.exponential(1/lp1,len(df)),3)

In [513]:
import pandas as pd
from lifelines import CoxPHFitter
pop = sim_population(N=10000, step_forward=5, randomseed=42)
df_cox = pop.history[0][['end', 'event_b', 'age', 'bmi', 'hyp', 'smoke', 'sex', 'eth']].copy()

In [514]:
cph = CoxPHFitter()
cph.fit(df_cox, duration_col='end', event_col='event_b')
s =cph.summary[['coef', 'se(coef)', 'p']]
round(s,4)

,coef,se(coef),p
covariate,,,
age,0.0267,0.0009,0.0000
bmi,0.0356,0.0152,0.0193
hyp,0.4815,0.0318,0.0000
smoke,-0.0112,0.0388,0.7718
sex,0.0256,0.0278,0.3577
eth,-0.0104,0.0208,0.6163


In [515]:
df1 = pop.history[2][['start','end', 'event_d', 'age', 'bmi', 'hyp', 'smoke', 'sex', 'eth']]
cph.fit(df1, duration_col='end', event_col='event_d', entry_col="start")
s1 =cph.summary[['coef', 'se(coef)', 'p']]
round(s1,4)

IndexError: list index out of range

In [ ]:
# Now check if going from 0 till the end and taking ever happening event_a returns similar coefficients: 
df1_first = pop.history[2][['first_e','end', 'age', 'bmi', 'hyp', 'smoke', 'sex', 'eth']].copy()
df1_first['event'] = df1_first['first_e'].notna().astype(int) # if a ever happened = 1
df1_first['time'] = df1_first['first_e'].where(df1_first['first_e'].notna(), 10) #time is from first_a column
cph.fit(df1_first[['time','event', 'age', 'bmi', 'hyp', 'smoke', 'sex', 'eth']], 
        duration_col='time', event_col='event')
s1_first = cph.summary[['coef', 'se(coef)', 'p']]
round(s1_first,4)

In [ ]:
df_short["event_a_1"] =  df_short['first_a'].notna().astype(int)
df_short['time'] = df_short['first_a'].where(df_short['first_a'].notna(), 12)
cph.fit(df_short[['time','event_a_1', 'age_start', 'bmi', 'hyp', 'smoke', 'sex', 'eth']], 
        duration_col='time', event_col='event_a_1')
sdfg_first = cph.summary[['coef', 'se(coef)', 'p']]
round(sdfg_first,4)

## 3 Define transformer

Embeddings: 1) Static context (time-invariant)   sex, eth; 2) Dynamic context (time-varying, per step)  age, bmi, hyp, smoke, 3) Token per step  {NONE, A, B, C, D, E}

Encodings (to sum together): 1) Event token embedding; 2) Dynamic covariate embedding; 3) Static covariate embedding (broadcast over time)

#### a) Shape df_long for transformer

"alive" is always the first token, no other features are required for it.

Only BMI is kept as time-varying.

All other covariates are time-invariant.

In [620]:
def build_event_sequences(df_long):
    """
    Build sequences for each patient with:
    - first token = "alive"
    - subsequent tokens = real events (a,b,c,...)
    """
    sequences = []
    for pid, g in df_long.groupby("id"):
        g = g.sort_values("time")
        # input tokens: first token = alive, then real events
        tokens = ["alive"] + g["event"].str.replace("first_", "", regex=False).tolist()
        # features: time-varying BMI only
        bmi = g[["bmi"]].to_numpy()
        # time gaps
        times = g["time"].to_numpy()
        # time-invariant covariates
        cov_invar = g.iloc[0][["age_start","hyp","smoke","sex","eth"]].to_numpy()
        
        sequences.append({
            "id": pid,
            "tokens": tokens,
            "bmi": bmi,
            "times": times,
            "cov_invar": cov_invar
        })
    return sequences


In [631]:
def pad_sequences(sequences, vocab, output_vocab):
    """
    Convert sequences to tensors, padding as needed.
    output_vocab excludes 'alive' (only real events are predicted)
    """
    token_seqs = []
    bmi_seqs = []
    dt_seqs = []
    cov_invar_list = []
    lengths = []

    for s in sequences:
        tokens = [vocab[t] for t in s["tokens"]]
        bmi = s["bmi"]
        times = s["times"]
        dt = np.diff(np.concatenate([[0.0], times]))
        token_seqs.append(tokens)
        bmi_seqs.append(bmi)
        dt_seqs.append(dt[:, None])
        cov_invar_list.append(s["cov_invar"])
        lengths.append(len(tokens))

    max_len = max(lengths)

    # pad token sequences
    X_tokens = torch.tensor(
        [seq + [vocab["alive"]] * (max_len - len(seq)) for seq in token_seqs],
        dtype=torch.long
    )
    # pad bmi sequences
    X_bmi = torch.tensor(
        np.stack([np.vstack([b, np.zeros((max_len - len(b), b.shape[1]))]) for b in bmi_seqs]),
        dtype=torch.float32
    )
    # pad dt
    X_dt = torch.tensor(
        np.stack([np.vstack([d, np.zeros((max_len - len(d),1))]) for d in dt_seqs]),
        dtype=torch.float32
    )
    # --- covariates ---
    # convert all cov_invar entries to float32 numpy arrays first
    cov_invar_list_float = [np.array(c, dtype=np.float32) for c in cov_invar_list]
    X_cov_invar = torch.tensor(np.stack(cov_invar_list_float), dtype=torch.float32)
    
    lengths = torch.tensor(lengths)

    return X_tokens, X_bmi, X_dt, X_cov_invar, lengths




#### b) Transformer 
* next-event prediction/Delphi2M style
* P(event_k | event_<k, covariates_<k)

In [623]:
class EventSequenceTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, d_model) #event tokens
        self.time_emb = nn.Linear(1, d_model)   # dt
        self.cov_invar_emb = nn.Linear(5, d_model)    # age, hyp, smoke, sex, eth
        self.cov_time_emb = nn.Linear(1, d_model)       #bmi, time-variant covariate
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True )
        
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.out = nn.Linear(d_model, vocab_size)
    
    def forward(self, token_ids, dt, cov_time, cov_invar):
        """ 
        token_ids: [B, T] event tokens
        dt: [B, T, 1] delta-times
        cov_time: [B, T, 1] time-varying covariates (BMI)
        cov_invar: [B, n_timeinv] baseline covariates
        """
        x = self.token_emb(token_ids) + self.time_emb(dt) + self.cov_time_emb(cov_time)
        
        # embed time-invariant covariates and broadcast
        cov_invar_emb = self.cov_invar_emb(cov_invar).unsqueeze(1)  # [B,1,d]
        
        x = x + cov_invar_emb  # broadcast along sequence
        
        x = self.encoder(x)
        logits = self.out(x)
        
        return logits

#### c) Loss

In [541]:
#STANDARD CROSS_ENTROPY LOSS

def masked_cross_entropy(logits, targets, lengths):
    """     logits:  [B, T-1, V]     targets: [B, T-1]     lengths: [B]   """
    B, T, V = logits.shape
    mask = ( torch.arange(T)[None, :] < (lengths - 1)[:, None])
    loss = torch.nn.functional.cross_entropy(
        logits.reshape(-1, V), targets.reshape(-1),  reduction="none" )
    loss = loss.view(B, T)
    
    return (loss * mask).sum() / mask.sum()

In [641]:
# ALTERNATIVE DELPHI-2M style loss
# mask "alive" in the loss
def masked_competing_exp_loss(logits, targets, dt, lengths, alpha=1.0, output_indices=None):
    """
    Compute masked competing event loss for transformer predictions.
    Assumes:
        logits: [B, T-1, vocab_size] (predicted for next tokens)
        targets: [B, T] (full token sequence including starting token)
        dt: [B, T, 1] (time gaps)
    """
    B, T_minus1, _ = logits.shape
    device = logits.device

    # Mask for valid steps (exclude starting token)
    mask = torch.arange(T_minus1, device=device)[None, :] < (lengths - 1)[:, None]

    # Only real events
    logits_real = logits[:, :, output_indices]  # [B, T-1, K_real]
    
    # Targets: shift by 1 to match logits
    targets_real = targets[:, 1:].clone()  # [B, T-1]

    # Map target token IDs to 0..K-1 relative to output_indices
    index_map = {vocab_id: i for i, vocab_id in enumerate(output_indices)}
    for b in range(B):
        for t_step in range(targets_real.shape[1]):
            token_id = targets_real[b, t_step].item()
            if token_id in index_map:
                targets_real[b, t_step] = index_map[token_id]
            else:
                # starting token or padding → mapped to 0 (ignored by mask)
                targets_real[b, t_step] = 0

    # Event loss
    loss_event = torch.nn.functional.cross_entropy(
        logits_real.reshape(-1, len(output_indices)),
        targets_real.reshape(-1),
        reduction="none"
    ).view(B, T_minus1)

    # Time loss
    dt_loss = dt[:, 1:, 0]  # aligned with logits
    log_lambda_sum = torch.logsumexp(logits_real, dim=-1)  # [B, T-1]
    lambda_sum = torch.exp(log_lambda_sum)
    loss_time = -log_lambda_sum + lambda_sum * dt_loss  # [B, T-1]

    # Combine losses with mask
    loss = (loss_event + alpha * loss_time) * mask
    return loss.sum() / mask.sum()



#### d) Training function

In [642]:
# TRAINING FUNCTION 

def train_transformer(
    model,
    sequences,
    vocab,
    n_epochs=10,
    lr=1e-3,
    batch_size=32,
    alpha=1.0,
    device="cpu",
    starting_token="alive"
):
    """
    Train EventSequenceTransformer with starting token automatically excluded from targets.
    """
    # Infer output vocab
    output_vocab = [e for e in vocab.keys() if e != starting_token]
    output_indices = [vocab[e] for e in output_vocab]

    model.to(device)
    model.train()

    # Build tensors
    X_tokens, X_bmi, X_dt, X_cov_invar, lengths = pad_sequences(sequences, vocab, output_vocab)
    X_tokens = X_tokens.to(device)
    X_bmi = X_bmi.to(device)
    X_dt = X_dt.to(device)
    X_cov_invar = X_cov_invar.to(device)
    lengths = lengths.to(device)

    N = X_tokens.size(0)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(n_epochs):
        perm = torch.randperm(N)
        total_loss = 0.0
        for i in range(0, N, batch_size):
            idx = perm[i:i+batch_size]
            tokens = X_tokens[idx]
            bmi = X_bmi[idx]
            dt = X_dt[idx]
            cov_invar = X_cov_invar[idx]
            lens = lengths[idx]

            optimizer.zero_grad()
            logits = model(tokens[:, :-1], bmi[:, :-1], dt[:, :-1], cov_invar)  # [B, T-1, vocab_size]
            loss = masked_competing_exp_loss(logits, tokens, dt, lens, alpha, output_indices)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)

        print(f"Epoch {epoch+1:03d} | loss = {total_loss / N:.4f}")




## 4 Fit transformer to data

In [638]:
#get sequences from df_long
dfdf = build_event_sequences(df_long)

In [629]:
dfdf[0]

{'id': 1,
 'tokens': ['alive', 'alive', 'b', 'c'],
 'bmi': array([[-1.8],
        [-1.8],
        [-1.8]]),
 'times': array([0.   , 0.059, 3.118]),
 'cov_invar': array([np.float64(62.1), np.int64(0), np.int64(0), np.int64(1),
        np.int64(0)], dtype=object)}

In [644]:
vocab = {"alive": 0, "a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
model = EventSequenceTransformer(vocab_size=len(vocab), d_model=64, nhead=4)
train_transformer(model, dfdf, vocab, lr=1e-3, batch_size=64, n_epochs=10)

Epoch 001 | loss = 2.4982
Epoch 002 | loss = 2.1722
Epoch 003 | loss = 2.1069
Epoch 004 | loss = 1.9087
Epoch 005 | loss = 2.3769
Epoch 006 | loss = 2.3550
Epoch 007 | loss = 2.2446
Epoch 008 | loss = 2.2183
Epoch 009 | loss = 2.0962
Epoch 010 | loss = 2.1059


In [647]:
train_transformer(model, dfdf, vocab, lr=1e-3, batch_size=256, n_epochs=5)

Epoch 001 | loss = 2.4042
Epoch 002 | loss = 2.3427
Epoch 003 | loss = 2.4829
Epoch 004 | loss = 2.4191
Epoch 005 | loss = 2.3404


## 5 Test transformer on df_long_test

### a) create test set

In [701]:
# create test set
pop_test = sim_population(N=20000, step_forward=2, randomseed=3456)
pop_test.step() # move +2y
pop_test.step() # move + 2y
pop_test.step() #move + 2y
pop_test.step() #move + 2y
pop_test.step() #move + 2y
event_cols = ["first_a", "first_b", "first_c", "first_d", "first_e"]
context_cols = ['age_start', 'bmi', 'hyp', 'smoke', 'sex', 'eth']
df_short_test = pop_test.history[5][["id"]+context_cols + event_cols]
df_short_test["first_alive"] = 0
df_long_test = df_short_test.melt(id_vars = ["id"] + context_cols, 
                       value_vars = event_cols + ["first_alive"], 
                       var_name = "event", 
                       value_name = "time").dropna(subset=["time"])
df_long_test["age"] = df_long_test["age_start"] + df_long_test["time"]
df_long_test = df_long_test.sort_values(by=['id', "time"])
df_long_test.head()
del pop_test
df_long_test.head()

C:\Users\dinab\AppData\Local\Temp\ipykernel_43840\191907525.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_short_test["first_alive"] = 0


,id,age_start,bmi,hyp,smoke,sex,eth,event,time,age
100000,1,44.4,0.6,0,0,1,0,first_alive,0.000,44.400
40000,1,44.4,0.6,0,0,1,0,first_c,6.685,51.085
0,1,44.4,0.6,0,0,1,0,first_a,8.774,53.174
20000,1,44.4,0.6,0,0,1,0,first_b,11.400,55.800
100001,2,49.3,-0.1,0,0,0,0,first_alive,0.000,49.300


In [ ]:
# probability event k happens within horizon tau (cum_haz from logit over time tau)
# P(T ≤ τ, event = k) = (λ_k / Λ) * (1 − exp(-Λ τ))
def cumulative_incidence(logits, tau):
    lambdas = torch.exp(logits)                # [B, T, K]
    lambda_sum = lambdas.sum(dim=-1, keepdim=True)
    return (lambdas / lambda_sum) * (1 - torch.exp(-lambda_sum * tau))

### b) Predict cause-specific hazards / cumilative incidences given initial context 

In [664]:

def predict_cumulative_incidence(
    model,    cov_invar,    vocab,    horizon,    dt_step=0.1,    bmi=None,    start_event="alive",    device="cpu"):
    """
    Predict cause-specific cumulative incidence functions (CIFs) over a time horizon.
    Assumes the first token is `start_event` (default "alive").
        Parameters
    ----------
    model : nn.Module         Trained EventSequenceTransformer
    cov_invar : list or array, shape [n_cov]         Time-invariant covariates (e.g., age, hyp, smoke, sex, eth)
    vocab : dict         Full vocab including starting token, e.g., {"alive": 0, "a":1, ...}
    horizon : float         Prediction horizon in same units as dt_step
    dt_step : float         Discretization step for horizon
    bmi : array-like, optional, shape [n_steps]         Time-varying BMI sequence. If None, assumed 0.
    start_event : str         Token for starting state ("alive" by default)
    device : str         "cpu" or "cuda"
      Returns
    -------
    dict : {event_name: CIF at horizon}
    """
    model.eval()
    device = torch.device(device)
    
    # Real events (exclude starting token)
    event_names = [e for e in vocab.keys() if e != start_event]
    event_indices = [vocab[e] for e in event_names]
    K = len(event_names)
    
    n_steps = int(np.ceil(horizon / dt_step))
    
    # --- Tokens ---
    start_token = vocab[start_event]
    tokens = torch.full((1, n_steps), start_token, dtype=torch.long, device=device)
    
    # --- dt array ---
    dt_array = torch.full((1, n_steps, 1), dt_step, dtype=torch.float32, device=device)
    
    # --- BMI ---
    if bmi is None:
        bmi_array = torch.zeros((1, n_steps, 1), dtype=torch.float32, device=device)
    else:
        bmi_seq = np.array(bmi, dtype=np.float32)
        if len(bmi_seq) != n_steps:
            # repeat or interpolate to n_steps
            bmi_seq = np.interp(
                np.linspace(0, len(bmi_seq)-1, n_steps),
                np.arange(len(bmi_seq)),
                bmi_seq
            )
        bmi_array = torch.tensor(bmi_seq, dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(-1)
    
    # --- cov_invar ---
    cov_array = torch.tensor(cov_invar, dtype=torch.float32, device=device).unsqueeze(0)
    
    # --- forward pass ---
    with torch.no_grad():
        logits = model(tokens, bmi_array, dt_array, cov_array)  # [1, T, vocab_size]
        # cause-specific hazards for real events
        hazards = torch.softmax(logits[:, :, event_indices], dim=-1)[0]  # [T, K]
    
    # --- CIF accumulation ---
    cum_incidence = torch.zeros(K, device=device)
    H_total = torch.tensor(0.0, device=device)   # <- key fix
    for t in range(n_steps):
        S_t = torch.exp(-H_total)
        lambda_t = hazards[t]
        dCIF = S_t * lambda_t * dt_step
        cum_incidence += dCIF
        H_total += lambda_t.sum() * dt_step

    S_final = torch.exp(-H_total)  # probability of remaining in "alive"
    total_event_prob = cum_incidence.sum().item()
    print("Total event probability:", total_event_prob)
    print("Probability still alive:", S_final.item())
    print("Sum + survival:", total_event_prob + S_final.item())  # should be ~1

    return {event_names[k]: cum_incidence[k].item() for k in range(K)}

In [ ]:
cov_invar = [50, 0, 0, 1, 2]    # age, hyp, smoke, sex, eth
bmi_seq = [ -1.8, -1.7, -1.5, -1.2, -1.0 ]  # example time-varying BMI
horizon = 1
cif_pred = predict_cumulative_incidence(model, cov_invar, vocab, horizon, dt_step=0.01, bmi=bmi_seq)
print(cif_pred)
print("Total event probability:", sum(cif_pred.values()))


Total event probability: 0.6352865695953369
Probability still alive: 0.3678796887397766
Sum + survival: 1.0031662583351135
{'a': 0.5176251530647278, 'b': 0.06866668909788132, 'c': 0.04799608886241913, 'd': 0.0003877645649481565, 'e': 0.0006108522065915167}
Total event probability: 0.6352865477965679


In [702]:
df_short_test

,id,age_start,bmi,hyp,smoke,sex,eth,first_a,first_b,first_c,first_d,first_e,first_alive
0,1,44.4,0.6,0,0,1,0,8.774,11.400,6.685,NaN,NaN,0
1,2,49.3,-0.1,0,0,0,0,1.352,7.242,NaN,NaN,NaN,0
2,3,51.7,0.3,0,1,0,1,3.650,4.896,0.436,NaN,6.094,0
3,4,54.5,1.4,1,1,1,2,1.314,0.754,2.384,NaN,NaN,0
4,5,55.1,1.9,0,0,1,1,1.311,0.232,0.150,3.345,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,19996,37.5,-0.1,0,0,1,0,NaN,6.804,NaN,NaN,NaN,0
19996,19997,30.2,-0.5,1,0,1,1,11.363,3.835,10.962,NaN,10.336,0
19997,19998,42.0,2.5,1,0,0,1,0.222,1.275,0.215,NaN,NaN,0
19998,19999,33.7,-2.2,0,0,0,1,2.224,0.179,1.725,NaN,NaN,0


You want to take df_long_test, extract only the initial row per patient (first_alive), compute the logits for all events from the model, and then add those logits as new columns to the DataFrame. Here’s a clean way to do that:

In [703]:

def add_initial_logits(df_short_test, model, vocab, device="cpu"):
    """
    For each patient, take the 'first_alive' row and compute logits for all events,
    then add logits as new columns to df_long_test.
    """
    model.eval()
    device = torch.device(device)
    
    all_logits = []
    
    with torch.no_grad():
        for idx, row in df_short_test.iterrows():
            # Prepare tensors
            token = torch.tensor([[vocab['alive']]], dtype=torch.long, device=device)   # [1,1]
            dt = torch.tensor([[[0.0]]], dtype=torch.float32, device=device)            # [1,1,1]
            bmi = torch.tensor([[[row['bmi']]]], dtype=torch.float32, device=device)    # [1,1,1]
            cov_invar = torch.tensor([[row['age_start'], row['hyp'], row['smoke'], row['sex'], row['eth']]],
                                     dtype=torch.float32, device=device)                 # [1, n_cov_invar]

            # Forward pass
            logits = model(token, bmi, dt, cov_invar)  # returns logits only
            logits = logits[0,0]                       # [vocab_size]

            # Keep only real events (exclude 'alive')
            logits_real = {f"logits_0_{e}": logits[vocab[e]].item() for e in vocab if e != "alive"}
            all_logits.append(logits_real)
    
    # Convert to DataFrame
    df_logits = pd.DataFrame(all_logits)
    df_logits["id"] = df_short_test["id"].values
    
    # Merge back into original df_long_test
    df_short_test_out = df_short_test.merge(df_logits, on = "id", how='left')
    
    return df_short_test_out



In [707]:
df_logits = add_initial_logits(df_short_test, model, vocab)
df_logits.head()


,id,age_start,bmi,hyp,smoke,sex,eth,first_a,first_b,first_c,first_d,first_e,first_alive,logits_0_a,logits_0_b,logits_0_c,logits_0_d,logits_0_e
0,1,44.4,0.6,0,0,1,0,8.774,11.400,6.685,NaN,NaN,0,4.214895,1.736503,1.380609,-3.584434,-3.131568
1,2,49.3,-0.1,0,0,0,0,1.352,7.242,NaN,NaN,NaN,0,4.057097,1.663436,1.318348,-3.556593,-3.116116
2,3,51.7,0.3,0,1,0,1,3.650,4.896,0.436,NaN,6.094,0,3.905173,1.835697,1.490658,-3.261499,-2.822488
3,4,54.5,1.4,1,1,1,2,1.314,0.754,2.384,NaN,NaN,0,3.655279,2.001981,1.663805,-2.908547,-2.475613
4,5,55.1,1.9,0,0,1,1,1.311,0.232,0.150,3.345,NaN,0,3.770021,1.770256,1.432246,-3.241734,-2.812965


#### C-index

In [743]:
import numpy as np
from lifelines.utils import concordance_index

def c_index_for_event(df_logits, event, horizon):
    """
    df_logits: wide dataframe (one row per subject)
    event: string, e.g. "a", "b", ...
    """
    time_col = f"first_{event}"
    risk_col = f"logits_0_{event}"
    T_event = df_logits[time_col]

    # observed event indicator
    event_observed = (T_event.notna()& (T_event <= horizon)).astype(int)
    print(sum(event_observed))
    # time to event or censoring
    other_events = [f"first_{e}" for e in ["a","b","c","d","e"] if e != event]
    censor_time = min(horizon, np.max(df_logits[["first_a", "first_b", "first_c","first_d", "first_e"]]))

    time = np.minimum(df_logits[time_col].fillna(censor_time).values, horizon)

    risk = df_logits[risk_col].values

    return concordance_index(time, risk, event_observed)


In [746]:
events = ["a", "b", "c", "d", "e"]
for e in events:
    cidx = c_index_for_event(df_logits, e, horizon = 1)
    print(f"C-index for event {e}: {cidx:.3f}")

3699
C-index for event a: 0.636
3746
C-index for event b: 0.619
3324
C-index for event c: 0.534
252
C-index for event d: 0.436
236
C-index for event e: 0.473


In [747]:
for e in events:
    cidx = c_index_for_event(df_logits, e, horizon = 10)
    print(f"C-index for event {e}: {cidx:.3f}")

15476
C-index for event a: 0.624
16523
C-index for event b: 0.595
13575
C-index for event c: 0.551
2697
C-index for event d: 0.449
2703
C-index for event e: 0.452
